# Merge CSVs with Common ID

In [1]:
# Importation des bibliothèques nécessaires
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Configuration pour afficher les graphiques dans le notebook
%matplotlib inline
plt.style.use('ggplot')
sns.set(style='whitegrid')

In [2]:
# Charger tous les fichiers CSV dans un dictionnaire de DataFrames
dataframes = {}

for csv_filename in ['Datasets/Tabular/Test/dataset1_with_target.csv', 'Datasets/Tabular/Test/dataset2_features_only.csv']:
    dataframes[csv_filename] = pd.read_csv(csv_filename)
    print(f"Table {csv_filename} shape: {dataframes[csv_filename].shape}")
    print(f"Column names: {dataframes[csv_filename].columns.tolist()}")

# Afficher la liste des DataFrames chargés
list(dataframes.keys())


Table Datasets/Tabular/Test/dataset1_with_target.csv shape: (1000, 11)
Column names: ['Feature_1', 'Feature_2', 'Feature_3', 'Feature_4', 'Feature_5', 'Feature_6', 'Feature_7', 'Feature_8', 'Feature_9', 'YTarget', 'ID']
Table Datasets/Tabular/Test/dataset2_features_only.csv shape: (1000, 10)
Column names: ['Extra_Feature_1', 'Extra_Feature_2', 'Extra_Feature_3', 'Extra_Feature_4', 'Extra_Feature_5', 'Extra_Feature_6', 'Extra_Feature_7', 'Extra_Feature_8', 'Extra_Feature_9', 'ID']


['Datasets/Tabular/Test/dataset1_with_target.csv',
 'Datasets/Tabular/Test/dataset2_features_only.csv']

In [3]:
# Fusionner tous les DataFrames sur la colonne d'ID commune
merged_df = None

for df_name, df in dataframes.items():
    if "ID" not in df.columns:
        print(f"Attention: La colonne 'ID' n'est pas présente dans {df_name}")
        continue
        
    if merged_df is None:
        merged_df = df.copy()
    else:
        # Utiliser un suffixe pour éviter la duplication des noms de colonnes
        merged_df = pd.merge(merged_df, df, on="ID", how='outer', 
                             suffixes=('', f'_{df_name}'))
# Standardize merged df
df = merged_df

# Afficher des informations sur le DataFrame fusionné
print("Shape du DataFrame final:", df.shape)
print("Colonnes du DataFrame final:", df.columns.tolist())
df.head()


Shape du DataFrame final: (1000, 20)
Colonnes du DataFrame final: ['Feature_1', 'Feature_2', 'Feature_3', 'Feature_4', 'Feature_5', 'Feature_6', 'Feature_7', 'Feature_8', 'Feature_9', 'YTarget', 'ID', 'Extra_Feature_1', 'Extra_Feature_2', 'Extra_Feature_3', 'Extra_Feature_4', 'Extra_Feature_5', 'Extra_Feature_6', 'Extra_Feature_7', 'Extra_Feature_8', 'Extra_Feature_9']


,Feature_1,Feature_2,Feature_3,Feature_4,Feature_5,Feature_6,Feature_7,Feature_8,Feature_9,YTarget,ID,Extra_Feature_1,Extra_Feature_2,Extra_Feature_3,Extra_Feature_4,Extra_Feature_5,Extra_Feature_6,Extra_Feature_7,Extra_Feature_8,Extra_Feature_9
0,0.855362,0.956623,-0.063627,-1.396212,1.414643,1.829330,-0.595423,0.017726,-0.036985,0,1,-0.974291,1.205792,-0.781847,-0.499613,0.426687,-0.439700,-0.487868,1.475964,0.517337
1,1.239500,2.891659,0.843971,0.793947,-1.786811,0.505111,1.840037,1.431398,2.100643,1,2,-0.296307,-0.762124,-0.259125,-0.442229,-1.498206,-0.247023,-4.713218,1.440885,1.791479
2,-2.318786,-1.940552,-0.006539,-0.279792,-0.774091,0.302895,-0.652285,-0.066638,-1.676665,0,3,-0.122152,-1.954993,0.137289,-0.514932,-0.138827,1.212155,1.306689,-1.044425,0.268705
3,-1.360879,-0.416040,1.694473,-0.253140,0.057397,0.413996,-0.992090,1.384977,-1.955474,0,4,1.618668,-2.110853,0.911275,-0.882978,0.258622,-0.162351,1.075656,-1.199623,-0.030579
4,0.928649,0.518915,-0.661731,0.654230,0.343298,-0.123298,0.803280,-1.395736,-0.737915,1,5,0.592358,0.545329,-0.847851,-1.166228,2.643939,1.182759,3.801083,-1.400315,-1.132902


# Random Forest

In [4]:
import xgboost as xgb
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score


In [5]:
target_column = "YTarget"
num_classes = 2

X_train, X_test, y_train, y_test = train_test_split(df.drop(columns=[target_column]), df[target_column], test_size=0.2, random_state=42)

# Create DMatrix for XGBoost
dtrain = xgb.DMatrix(X_train, label=y_train)
dtest = xgb.DMatrix(X_test, label=y_test)

# Set parameters for XGBoost
params = {
    'objective': 'multi:softmax',
    'num_class': num_classes,
    'max_depth': 6,
    'eta': 0.3,
    'eval_metric': 'mlogloss'
}

# Train the model
num_rounds = 100
model = xgb.train(params, dtrain, num_rounds)

# Make predictions
y_pred = model.predict(dtest)

# Evaluate the model
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy}")

# Make predictions on the test set
y_pred = model.predict(dtest)

# Evaluate the model
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy}")


Accuracy: 0.955
Accuracy: 0.955
